# 03 — Rapport par appareil, étape par étape

Ce notebook **reproduit chaque section** de la page produite par
`report_builder.build_rapport_appareil`, avec les **graphiques inline**, pour l'appareil
`SEIN` au niveau `AP-HP` (exemple du notebook). La vue page complète (IFrame) est conservée
à la fin.

> ⚠ Duplique volontairement la logique de `build_rapport_appareil` (branche `entity == 'AP-HP'`).
> Re-synchroniser si `report_builder` évolue.

In [ ]:
# Parametres papermill
APPAREIL = 'SEIN'
ENTITY   = 'AP-HP'

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from pathlib import Path
import pandas as pd
from IPython.display import HTML

DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Mode RÉEL : aligne le bandeau des pages embarquées (pas de « Données simulées »).
from report_builder import set_fake_data
set_fake_data(False)

from report_builder import load_aphp, load_regional, load_survival, organe_counts_table
from chart_utils import (
    line_evolution, donut_market_share, regional_comparison, treemap_organes,
    survival_by_stage, survival_evolution, delay_evolution, delay_comparison_bar,
    TREATMENT_COLS, GHU_LIST, REGIONAL_COLORS,
)

aphp = load_aphp(DATA_DIR)
reg  = load_regional(DATA_DIR)
surv = load_survival(DATA_DIR)

# Sous-ensembles (identiques a build_rapport_appareil)
app_data = aphp[(aphp.appareil == APPAREIL) & (aphp.organe == 'TOTAL')]
ent_app  = app_data[app_data.entite == ENTITY]
aphp_app = app_data[app_data.entite == 'AP-HP']
ghu_app  = app_data[app_data.entite.isin(GHU_LIST)]

years = sorted(ent_app.annee.unique())
last_year, prev_year = years[-1], years[-2]
lv = ent_app[ent_app.annee == last_year].iloc[0]
pv = ent_app[ent_app.annee == prev_year].iloc[0]
print(f'Appareil={APPAREIL} entity={ENTITY} - annees {years} - derniere={last_year}')

## Section 1 — Indicateurs clés

5 à 6 cartes KPI (incl. délai médian si présent). Vue tableau équivalente :

In [ ]:
mesures = [
    (f'Patients {ENTITY}',     'nb_patients'),
    ('Nouveaux patients',      'nb_nouveaux_patients'),
    ('Sejours chirurgie',      'nb_sejours_chirurgie'),
    ('Sejours chimiotherapie', 'nb_sejours_chimiotherapie'),
    ('Sejours radiotherapie',  'nb_sejours_radiotherapie'),
]
if pd.notna(lv.get('delai_global_median')):
    mesures.append(('Delai median (j)', 'delai_global_median'))
pd.DataFrame([
    {'Indicateur': lab, str(prev_year): int(pv[col]), str(last_year): int(lv[col]),
     'Var. %': round((lv[col]-pv[col])/pv[col]*100, 1) if pv[col] else None}
    for lab, col in mesures
])

## Section 2 — Évolution du nombre de patients

Branche `entity == 'AP-HP'` : le contexte est **régional**.
`regional_comparison` + `donut_market_share` (régional, dernière année) + séjours par mode de PEC.

In [ ]:
reg_app = reg[(reg.appareil == APPAREIL) & (reg.organe == 'TOTAL')]
fig_reg = regional_comparison(reg_app, 'nb_patients',
                              f'Contexte regional - {APPAREIL}',
                              color_map=REGIONAL_COLORS)
fig_reg.show()

In [ ]:
reg_app_last = reg_app[reg_app.annee == last_year]
fig_reg_donut = donut_market_share(
    reg_app_last, 'entite', 'nb_patients',
    f'Parts de marche regional - {APPAREIL} ({last_year})',
    entities=sorted(reg_app_last['entite'].unique()), color_map=REGIONAL_COLORS)
fig_reg_donut.show()

In [ ]:
melted = ent_app.melt(id_vars=['annee'], value_vars=list(TREATMENT_COLS.keys()),
                      var_name='type_sejour', value_name='nb_sejours')
melted['label'] = melted['type_sejour'].map(TREATMENT_COLS)
fig_sejours = line_evolution(melted, 'annee', 'nb_sejours', 'label',
                             f'Sejours par mode de PEC - {APPAREIL}',
                             entities=list(TREATMENT_COLS.values()))
fig_sejours.show()

## Section 3 — Analyse par organe

Tableau chiffré `organe_counts_table` (HTML) + `treemap_organes` (dernière année).

In [ ]:
HTML(organe_counts_table(aphp, ENTITY, APPAREIL))

In [ ]:
fig_tree = treemap_organes(aphp, ENTITY, APPAREIL, last_year)
fig_tree.show()

## Section 4 — Survie par stade

`survival_by_stage` (dernière année) + `survival_evolution` (stade I-III).

In [ ]:
fig_surv = survival_by_stage(surv, ENTITY, APPAREIL, year=last_year)
fig_surv.show()

In [ ]:
fig_surv_evo = survival_evolution(surv, ENTITY, APPAREIL, stade='I-III')
fig_surv_evo.show()

## Section 5 — Délais de prise en charge

`delay_evolution` + `delay_comparison_bar` (AP-HP + GHU, dernière année).

In [ ]:
fig_delay = delay_evolution(aphp, ENTITY, APPAREIL)
fig_delay.show()

In [ ]:
all_delay_ents = pd.concat([aphp_app, ghu_app])
fig_delay_cmp = delay_comparison_bar(all_delay_ents, APPAREIL, last_year)
fig_delay_cmp.show()

## Page complète (référence)

In [ ]:
from report_builder import build_rapport_appareil

out = build_rapport_appareil(APPAREIL, DATA_DIR, OUTPUT_DIR, entity=ENTITY)
print(f'Rapport genere : {out}')

from IPython.display import IFrame
IFrame(os.path.relpath(out), width='100%', height=800)